#### This script contains all the SQL statements required to populate the scorecard for the iplt20.com website

In [0]:
%sql
USE CATALOG ipl_2024_project;
USE SCHEMA pyspark;

#### Reference diag -->

![image_1780980862616.png](./image_1780980862616.png "image_1780980862616.png")

![image_1781069570919.png](./image_1781069570919.png "image_1781069570919.png")

#### **Match level information Required**
###### 1. Match Number
###### 2. Venue
###### 3. Match Date
###### 4. Winning Team Name

In [0]:
venue = spark.read.table("venue")
match = spark.read.table("match")
team = spark.read.table("team")
result = spark.read.table("result")

In [0]:
match_details = match.join(
    venue,
    match.venue_id == venue.venue_id,
    "inner"
).select(
    match.match_id,
    match.match_date,
    venue.venue_name
)

result_details = result.join(
    team,
    result.winning_team_id == team.team_id,
    "inner"
).join(
    match_details,
    result.match_id == match_details.match_id,
    "inner"
).select(
    match_details.match_id,
    match_details.match_date,
    match_details.venue_name,
    team.team_name.alias("winning_team_name")
)

match_overview = result_details
match_overview.display()

#### **Innings level information Required**
###### 1. Name of the teams played
###### 2. Total runs scored
###### 3. Total wickets lost
###### 4. Total overs played

In [0]:
innings = spark.read.table("innings")
score_by_ball = spark.read.table("score_by_ball")

In [0]:
from pyspark.sql.functions import sum, col, count, concat, lit, when
i = innings.alias("i")
t = team.alias("t")
s = score_by_ball.alias("s")

innings_overview = i.join(
    t,
    i.batting_team_id == t.team_id,
    "inner"
).join(
    s,
    (s.match_id == i.match_id) &
    (s.innings_no == i.innings_no),
    "inner"
).groupBy(
    i.match_id,
    i.innings_no,
    t.team_name
).agg(
    sum(s.runs_off_bat + s.extras).alias("total_runs"),
    count(s.wicket_type).alias("total_wickets"),
    # count(s.ball_no).alias("total_balls"),
    # count(s.wides).alias("wides_count"),
    # count(s.noballs).alias("noballs_count"),
    # (count(s.ball_no) - count(s.wides)).alias("good_balls"),
    # (((count(s.ball_no) - count(s.wides)) / 6).cast("integer")).alias("overs"),
    # (((count(s.ball_no) - count(s.wides))) % 6).cast("integer").alias("remaining_balls"),
    when (
        (((count(s.ball_no) - count(s.wides))) % 6).cast("integer") == 0,
        (((count(s.ball_no) - count(s.wides)) / 6).cast("integer")).cast("string")
    ).otherwise
    (
        concat(
            (((count(s.ball_no) - count(s.wides)) / 6).cast("integer")),
            lit('.'),
            (((count(s.ball_no) - count(s.wides))) % 6).cast("integer")
        )
    ).alias("overs_new")
)

innings_overview.display()